# Part A Question 3: POS Taggers and Syntactic Analysis

This notebook demonstrates POS tagging using the NLTK POS tagger, TextBlob POS tagger, and a regular-expression tagger. It also builds a parse tree for `Data_2.txt`.

In [1]:
import string
from pathlib import Path
from pprint import pprint
from typing import Any, cast

from nltk import CFG, ChartParser, RegexpTagger, pos_tag
from nltk.tokenize import TreebankWordTokenizer
from textblob import TextBlob

In [2]:
DATA_PATH = Path('data/Data_2.txt')
if not DATA_PATH.exists():
    DATA_PATH = Path('../data/Data_2.txt')


def load_text(path: Path) -> str:
    """Read a UTF-8 text file and normalize whitespace."""

    return ' '.join(path.read_text(encoding='utf-8').split())


text = load_text(DATA_PATH)
text

'The big black dog barked at the white cat and chased away.'

## 1. Tokenization

In [3]:
def word_tokens(text: str) -> list[str]:
    """Return word tokens without punctuation-only marks."""

    tokenizer = TreebankWordTokenizer()
    return [token for token in tokenizer.tokenize(text) if token not in string.punctuation]


tokens = word_tokens(text)
pprint(tokens)

['The',
 'big',
 'black',
 'dog',
 'barked',
 'at',
 'the',
 'white',
 'cat',
 'and',
 'chased',
 'away']


## 2. NLTK POS Tagger

In [4]:
nltk_tags = pos_tag(tokens)
pprint(nltk_tags)

[('The', 'DT'),
 ('big', 'JJ'),
 ('black', 'JJ'),
 ('dog', 'NN'),
 ('barked', 'VBD'),
 ('at', 'IN'),
 ('the', 'DT'),
 ('white', 'JJ'),
 ('cat', 'NN'),
 ('and', 'CC'),
 ('chased', 'VBD'),
 ('away', 'RB')]


## 3. TextBlob POS Tagger

In [5]:
textblob_tags = [(str(word), str(tag)) for word, tag in cast(list[tuple[Any, Any]], TextBlob(text).tags)]
pprint(textblob_tags)

[('The', 'DT'),
 ('big', 'JJ'),
 ('black', 'JJ'),
 ('dog', 'NN'),
 ('barked', 'VBD'),
 ('at', 'IN'),
 ('the', 'DT'),
 ('white', 'JJ'),
 ('cat', 'NN'),
 ('and', 'CC'),
 ('chased', 'VBD'),
 ('away', 'RB')]


## 4. Regular Expression Tagger

In [6]:
regex_tagger = RegexpTagger(
    [
        (r'(?i)^(the|a|an)$', 'DT'),
        (r'(?i)^(and|or|but)$', 'CC'),
        (r'(?i)^(at|in|on|by|for|from|to|with)$', 'IN'),
        (r'(?i)^.*ed$', 'VBD'),
        (r'(?i)^(big|black|white)$', 'JJ'),
        (r'(?i)^away$', 'RB'),
        (r'(?i)^.*$', 'NN'),
    ]
)
regex_tags = regex_tagger.tag(tokens)
pprint(regex_tags)

[('The', 'DT'),
 ('big', 'JJ'),
 ('black', 'JJ'),
 ('dog', 'NN'),
 ('barked', 'VBD'),
 ('at', 'IN'),
 ('the', 'DT'),
 ('white', 'JJ'),
 ('cat', 'NN'),
 ('and', 'CC'),
 ('chased', 'VBD'),
 ('away', 'RB')]


## 5. Parse Tree

In [7]:
grammar = CFG.fromstring(
    """
    S -> NP VP
    NP -> Det AdjSeq N
    AdjSeq -> Adj Adj | Adj
    VP -> VP CC VP | VBD PP | VBD Adv
    PP -> P NP
    Det -> 'The' | 'the'
    Adj -> 'big' | 'black' | 'white'
    N -> 'dog' | 'cat'
    VBD -> 'barked' | 'chased'
    P -> 'at'
    CC -> 'and'
    Adv -> 'away'
    """
)

parser = ChartParser(grammar)
parse_trees = list(parser.parse(tokens))

for tree in parse_trees:
    print(tree.pformat(margin=100))

(S
  (NP (Det The) (AdjSeq (Adj big) (Adj black)) (N dog))
  (VP
    (VP (VBD barked) (PP (P at) (NP (Det the) (AdjSeq (Adj white)) (N cat))))
    (CC and)
    (VP (VBD chased) (Adv away))))
